# 04 · Embodied — a VLM plays a video game

`process(EnvironmentStep)` drives a closed loop: render a frame, the model picks a move, step, repeat. We compare an oracle, a random null, and (optionally) a real VLM on identical boards.

> Run on EC2 (GPU + Brain-Score data). Do **not** run on a laptop.


In [ ]:
from brainscore.harnesses.grid_game import (GridGameEnv, play_game,
    greedy_oracle_policy, random_action_policy, evaluate_policy)
from brainscore_core.model_interface import BrainScoreModel
from brainscore.model_helpers.policy_wrapper import PolicyWrapper
def make_model(policy):
    return BrainScoreModel('player', None, {}, {}, None,
                           action_fn=PolicyWrapper(policy, max_history=4))

## Oracle vs random null over 25 boards

In [ ]:
mk = lambda p: make_model(p)
oracle = evaluate_policy(mk, greedy_oracle_policy, n_episodes=25)
rand = evaluate_policy(mk, random_action_policy(0), n_episodes=25)
print('oracle:', oracle)
print('random:', rand)

## A single rendered frame (what a VLM policy sees)

In [ ]:
import matplotlib.pyplot as plt
env = GridGameEnv(size=5, seed=3)
obs = env.reset()
plt.imshow(obs['frame']); plt.axis('off'); plt.title('grid game frame')
plt.savefig('frame.png', dpi=120, bbox_inches='tight')
print(obs['ascii'])

**Scaling result (from `scripts/vlm_game`):** random 0.20 → Qwen2.5-VL-3B 0.0 → Qwen2.5-VL-7B 0.13 (efficiency 1.0 when it solves) → thinking-model-over-ASCII (perfect perception) → oracle 1.0. The visual VLMs' struggle vs the thinking model's success isolates *perception* as the bottleneck — the kind of decomposition the interface makes easy.